# London Fire Brigade - Response Time Regression

Predict `FirstPumpArriving_AttendanceTime` (seconds). Right-skewed target with median around 296 s and p95 around 588 s. We train Linear Regression, Random Forest and XGBoost using log1p transform on the target. Final reporting metrics (RMSE, MAE, R2) are computed back on the original second scale.

Strategy:
- Load `data/lfb_incidents_2009_2017.csv` (about 988k rows, 39 columns).
- Subsample to 500k rows for memory safety (random seed=42), as the project brief allows.
- Drop redacted/near-empty columns (`Postcode_full`, exact `Easting_m`, `Northing_m`, `Latitude`, `Longitude`, `UPRN`).
- Engineer time features (hour, weekday, month) and lightweight spatial proxies from rounded eastings/northings.
- Train/val/test 70/15/15 split, stratified by `IncidentGroup`.
- Save best model and metrics.

In [ ]:
import os, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import joblib

warnings.filterwarnings('ignore')
ROOT = Path('/root/AI/project_root')
DATA = ROOT / 'data' / 'lfb_incidents_2009_2017.csv'
DELIV = ROOT / 'deliverables'
DELIV.mkdir(exist_ok=True)
RNG = 42
print('Files OK:', DATA.exists())

## 1. Load CSV (typed) and subsample to 500k

In [ ]:
USECOLS = [
    'IncidentNumber', 'DateOfCall', 'CalYear', 'HourOfCall',
    'IncidentGroup', 'StopCodeDescription', 'PropertyCategory',
    'AddressQualifier', 'Postcode_district',
    'IncGeo_BoroughName', 'IncidentStationGround',
    'Easting_rounded', 'Northing_rounded',
    'FirstPumpArriving_AttendanceTime',
    'FirstPumpArriving_DeployedFromStation',
    'NumStationsWithPumpsAttending', 'NumPumpsAttending',
    'NumCalls'
]

DTYPES = {
    'CalYear': 'Int16', 'HourOfCall': 'Int8',
    'IncidentGroup': 'category', 'StopCodeDescription': 'category',
    'PropertyCategory': 'category', 'AddressQualifier': 'category',
    'Postcode_district': 'category', 'IncGeo_BoroughName': 'category',
    'IncidentStationGround': 'category',
    'FirstPumpArriving_DeployedFromStation': 'category',
    'Easting_rounded': 'float32', 'Northing_rounded': 'float32',
    'FirstPumpArriving_AttendanceTime': 'float32',
    'NumStationsWithPumpsAttending': 'float32',
    'NumPumpsAttending': 'float32', 'NumCalls': 'float32'
}

t0 = time.time()
df = pd.read_csv(DATA, usecols=USECOLS, dtype=DTYPES, encoding='utf-8-sig', low_memory=False)
print(f'Loaded {len(df):,} rows in {time.time()-t0:.1f}s, mem={df.memory_usage(deep=True).sum()/1e6:.0f} MB')
print('Target missing:', df['FirstPumpArriving_AttendanceTime'].isna().sum())

In [ ]:
# Drop rows where target is missing, then subsample to 500k
df = df.dropna(subset=['FirstPumpArriving_AttendanceTime']).reset_index(drop=True)
print('After dropna(target):', len(df))

if len(df) > 500_000:
    df = df.sample(n=500_000, random_state=RNG).reset_index(drop=True)
    print('Subsampled to 500k rows for memory safety')

y_secs = df['FirstPumpArriving_AttendanceTime'].astype('float32').values
print('Target stats:', pd.Series(y_secs).describe()[['mean','50%','min','max']].to_dict())

## 2. Feature engineering

Time features from `DateOfCall` + `HourOfCall`, spatial proxies from rounded eastings/northings (distance from a central reference point, treated as a crude proxy for distance to central London station coverage).

In [ ]:
df['DateOfCall'] = pd.to_datetime(df['DateOfCall'], dayfirst=True, errors='coerce')
df['weekday'] = df['DateOfCall'].dt.weekday.astype('Int8')
df['month'] = df['DateOfCall'].dt.month.astype('Int8')
df['day_of_year'] = df['DateOfCall'].dt.dayofyear.astype('Int16')
df['is_weekend'] = (df['weekday'] >= 5).astype('Int8')

# Hour bins (rush hour, night, etc.)
df['hour_sin'] = np.sin(2 * np.pi * df['HourOfCall'] / 24).astype('float32')
df['hour_cos'] = np.cos(2 * np.pi * df['HourOfCall'] / 24).astype('float32')

# Spatial proxy: distance from approximate central London (Trafalgar Square, OSGB36 Easting=530000, Northing=180400)
REF_E, REF_N = 530000.0, 180400.0
df['dist_central_m'] = np.sqrt(
    (df['Easting_rounded'] - REF_E)**2 + (df['Northing_rounded'] - REF_N)**2
).astype('float32')
df['dist_central_km'] = (df['dist_central_m'] / 1000).astype('float32')

# Pumps available (operational)
df['NumPumpsAttending'] = df['NumPumpsAttending'].fillna(1).astype('float32')
df['NumStationsWithPumpsAttending'] = df['NumStationsWithPumpsAttending'].fillna(1).astype('float32')
df['NumCalls'] = df['NumCalls'].fillna(1).astype('float32')

print('Engineered features added')

In [ ]:
# Final feature columns
NUM_COLS = ['CalYear', 'HourOfCall', 'weekday', 'month', 'day_of_year', 'is_weekend',
            'hour_sin', 'hour_cos', 'dist_central_km',
            'NumPumpsAttending', 'NumStationsWithPumpsAttending', 'NumCalls',
            'Easting_rounded', 'Northing_rounded']

CAT_COLS = ['IncidentGroup', 'StopCodeDescription', 'PropertyCategory',
            'AddressQualifier', 'IncGeo_BoroughName', 'IncidentStationGround',
            'FirstPumpArriving_DeployedFromStation']

# Fill NA in numerics with median
for c in NUM_COLS:
    if df[c].isna().any():
        df[c] = df[c].fillna(df[c].median())

# Strip target before splitting
y = df['FirstPumpArriving_AttendanceTime'].astype('float32').values
strat = df['IncidentGroup'].astype(str).fillna('Unknown')

X = df[NUM_COLS + CAT_COLS].copy()
for c in CAT_COLS:
    X[c] = X[c].astype('category')

print('X shape:', X.shape, '| y shape:', y.shape)

## 3. Train / val / test split (stratified by IncidentGroup)

In [ ]:
X_trainval, X_test, y_trainval, y_test, s_trainval, _ = train_test_split(
    X, y, strat, test_size=0.15, random_state=RNG, stratify=strat)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.1765, random_state=RNG, stratify=s_trainval)
# 0.1765 of the 85% trainval gives roughly 15% val of total
print('Train:', len(X_train), 'Val:', len(X_val), 'Test:', len(X_test))

# Log1p target for the regression head
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)
y_test_log = np.log1p(y_test)

In [ ]:
def eval_seconds(y_true_secs, y_pred_secs, label=''):
    rmse = float(np.sqrt(mean_squared_error(y_true_secs, y_pred_secs)))
    mae = float(mean_absolute_error(y_true_secs, y_pred_secs))
    r2 = float(r2_score(y_true_secs, y_pred_secs))
    print(f'{label:<30s} RMSE={rmse:7.2f}s  MAE={mae:7.2f}s  R2={r2:.4f}')
    return {'rmse_seconds': rmse, 'mae_seconds': mae, 'r2': r2}

results = {}

## 4. Baseline 1: Linear Regression with one-hot encoding

We restrict to a one-hot of `IncidentGroup` plus all numerics for tractable Linear Regression. High-cardinality categoricals (`StopCodeDescription`, station ground, postcode-like) are not one-hot encoded for the linear model, since that would blow up the feature dimension.

In [ ]:
LINEAR_CATS = ['IncidentGroup', 'PropertyCategory', 'AddressQualifier']
X_lin_train = pd.get_dummies(X_train[NUM_COLS + LINEAR_CATS], columns=LINEAR_CATS, drop_first=True, dtype='float32')
X_lin_val   = pd.get_dummies(X_val[NUM_COLS + LINEAR_CATS],   columns=LINEAR_CATS, drop_first=True, dtype='float32')
X_lin_test  = pd.get_dummies(X_test[NUM_COLS + LINEAR_CATS],  columns=LINEAR_CATS, drop_first=True, dtype='float32')
X_lin_val = X_lin_val.reindex(columns=X_lin_train.columns, fill_value=0)
X_lin_test = X_lin_test.reindex(columns=X_lin_train.columns, fill_value=0)

lin = LinearRegression()
t0 = time.time()
lin.fit(X_lin_train, y_train_log)
print(f'LinReg fit in {time.time()-t0:.1f}s')
pred_test_secs = np.expm1(lin.predict(X_lin_test))
results['linreg'] = eval_seconds(y_test, pred_test_secs, 'Linear Regression')

## 5. Baseline 2: Random Forest

Random Forest natively handles non-linearity. We label-encode high-cardinality categoricals to integer codes (RF in sklearn does not consume pandas categories, but integer codes work as ordinal proxies and trees split happily on them).

In [ ]:
def encode_cats(X_in):
    out = X_in.copy()
    for c in CAT_COLS:
        out[c] = out[c].cat.codes.astype('int32')
    return out

X_tree_train = encode_cats(X_train)
X_tree_val   = encode_cats(X_val)
X_tree_test  = encode_cats(X_test)

rf = RandomForestRegressor(
    n_estimators=120, max_depth=22, min_samples_leaf=20,
    n_jobs=-1, random_state=RNG)
t0 = time.time()
rf.fit(X_tree_train, y_train_log)
print(f'RF fit in {time.time()-t0:.1f}s')
pred_test_secs = np.expm1(rf.predict(X_tree_test))
results['random_forest'] = eval_seconds(y_test, pred_test_secs, 'Random Forest')

# Feature importances
rf_imp = pd.DataFrame({'feature': X_tree_train.columns, 'importance': rf.feature_importances_}).sort_values('importance', ascending=False)
print('Top 10 RF features:')
print(rf_imp.head(10).to_string(index=False))

## 6. Improved: XGBoost (native categorical, log target, more trees)

In [ ]:
# XGBoost native categorical handling: pass DataFrame with category dtypes, enable_categorical=True
xgb_params = dict(
    n_estimators=600, max_depth=8, learning_rate=0.06,
    subsample=0.85, colsample_bytree=0.85,
    min_child_weight=10, reg_lambda=1.0,
    objective='reg:squarederror', tree_method='hist',
    enable_categorical=True, n_jobs=-1, random_state=RNG)

xgb_model = xgb.XGBRegressor(**xgb_params)
t0 = time.time()
xgb_model.fit(
    X_train, y_train_log,
    eval_set=[(X_val, y_val_log)],
    verbose=False)
print(f'XGBoost fit in {time.time()-t0:.1f}s | best iter: {xgb_model.best_iteration if hasattr(xgb_model, "best_iteration") else "n/a"}')
pred_test_secs = np.expm1(xgb_model.predict(X_test))
results['xgboost'] = eval_seconds(y_test, pred_test_secs, 'XGBoost (log target)')

In [ ]:
# Feature importances (XGBoost gain)
imp = xgb_model.feature_importances_
xgb_imp = pd.DataFrame({'feature': X_train.columns, 'importance': imp}).sort_values('importance', ascending=False)
print('Top 12 XGBoost features:')
print(xgb_imp.head(12).to_string(index=False))
results['xgb_top_features'] = xgb_imp.head(12).to_dict(orient='records')

## 7. Lightweight hyperparameter search (random, 8 trials on a 100k subsample)

We do a small randomised search on a 100k subsample to keep wall time reasonable, then refit the winning config on the full train set.

In [ ]:
rng_search = np.random.default_rng(RNG)
search_grid = {
    'max_depth': [6, 7, 8, 9, 10],
    'learning_rate': [0.03, 0.05, 0.07, 0.1],
    'min_child_weight': [5, 10, 20, 40],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

# Subsample 100k for search
sub_idx = rng_search.choice(len(X_train), size=min(100_000, len(X_train)), replace=False)
X_sub = X_train.iloc[sub_idx]
y_sub_log = y_train_log[sub_idx]

best_cfg, best_score = None, np.inf
trials = []
for trial in range(8):
    cfg = {k: rng_search.choice(v).item() for k, v in search_grid.items()}
    m = xgb.XGBRegressor(
        n_estimators=400, **cfg,
        objective='reg:squarederror', tree_method='hist',
        enable_categorical=True, n_jobs=-1, random_state=RNG)
    m.fit(X_sub, y_sub_log, eval_set=[(X_val, y_val_log)], verbose=False)
    pred_val_secs = np.expm1(m.predict(X_val))
    rmse_v = float(np.sqrt(mean_squared_error(y_val, pred_val_secs)))
    trials.append({'cfg': cfg, 'val_rmse_seconds': rmse_v})
    if rmse_v < best_score:
        best_score, best_cfg = rmse_v, cfg
    print(f'trial {trial+1}: RMSE={rmse_v:.2f}s | cfg={cfg}')

print('\\nBest cfg:', best_cfg, 'val RMSE:', round(best_score, 2), 's')
results['hp_search_best_cfg'] = best_cfg
results['hp_search_trials'] = trials

In [ ]:
# Refit XGBoost with best_cfg on full train set, more trees
xgb_final = xgb.XGBRegressor(
    n_estimators=900, **best_cfg,
    objective='reg:squarederror', tree_method='hist',
    enable_categorical=True, n_jobs=-1, random_state=RNG)
t0 = time.time()
xgb_final.fit(X_train, y_train_log, eval_set=[(X_val, y_val_log)], verbose=False)
print(f'XGBoost final fit in {time.time()-t0:.1f}s')

pred_val_secs = np.expm1(xgb_final.predict(X_val))
pred_test_secs = np.expm1(xgb_final.predict(X_test))
results['xgboost_tuned_val'] = eval_seconds(y_val, pred_val_secs, 'XGBoost tuned (val)')
results['xgboost_tuned_test'] = eval_seconds(y_test, pred_test_secs, 'XGBoost tuned (test)')

## 8. Save artefacts

In [ ]:
# Save pickled XGBoost model
model_path = DELIV / 'lfb_xgboost.pkl'
joblib.dump({'model': xgb_final, 'feature_columns': list(X_train.columns), 'cat_cols': CAT_COLS, 'best_cfg': best_cfg}, model_path)
print('Saved model:', model_path)

metrics_path = DELIV / 'metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)
print('Saved metrics:', metrics_path)

print('\\nSummary of metrics on TEST set (original seconds):')
for k in ['linreg', 'random_forest', 'xgboost', 'xgboost_tuned_test']:
    if k in results:
        m = results[k]
        print(f'  {k:<22s}  RMSE={m["rmse_seconds"]:.2f}s  MAE={m["mae_seconds"]:.2f}s  R2={m["r2"]:.4f}')